# Experiment 1.3b-i-A: Feedback Measure -- Does sigma_1(W) Growth Rate Differ Between SGD and Muon?

## Motivation

In standard SGD with momentum, gradient updates are not spectrally uniform. The gradient of a
deep linear network naturally aligns with the dominant singular direction of the weight matrix.
This creates a **positive feedback loop**: a large leading singular value sigma_1(W) biases the
gradient to point along the same direction, which in turn amplifies sigma_1(W) further. Over
many steps, this self-reinforcing cycle is expected to produce **exponential growth** of the
top singular value: sigma_1(W) ~ exp(a * t).

Muon disrupts this mechanism. Its Newton-Schulz orthogonalization step projects the raw gradient
onto the nearest orthogonal matrix before applying momentum. An orthogonal matrix has all singular
values equal to 1, so the update magnitude is **spectrally flat** -- every direction receives the
same step size. This should **break the feedback loop**: without preferential amplification of the
dominant direction, sigma_1(W) can only grow at most linearly (bounded step per iteration), and
in practice should grow sub-linearly as the loss decreases.

## Hypothesis

| Quantity | SGD Prediction | Muon Prediction |
|----------|---------------|-----------------|
| log(sigma_1(W)) vs t | **Linear** (exponential growth) | **Sub-linear** (polynomial/bounded growth) |
| Best-fit model for sigma_1(W) | Exponential: R^2_exp > R^2_poly | Polynomial: R^2_poly > R^2_exp |
| corr(sigma_1(G), sigma_1(W)) | **High** (gradient tracks weight anisotropy) | **Low** (orthogonalization decorrelates) |
| Gini coefficient of SV spectrum | **Higher** (concentrated spectrum) | **Lower** (more uniform spectrum) |
| Condition number sigma_1/sigma_n | **Growing** (diverging spread) | **Bounded** (controlled spread) |

## Methodology

- **Architecture**: 4-layer deep linear network, 32x32 weight matrices, initialized near identity (I + 0.1*N(0,1))
- **Task**: Quadratic loss matching a fixed random target matrix W_target
- **Optimizers**: SGD with momentum (0.9) vs Muon (Newton-Schulz orthogonalized gradients + momentum 0.9)
- **Duration**: 500 training steps with fixed batch
- **Measurements at every step**:
  - sigma_1(W_i) and sigma_n(W_i) for each layer i (top and bottom singular values)
  - sigma_1(G_i) for each layer i (top singular value of the raw gradient)
  - Training loss
- **Post-hoc analysis**:
  - Fit exponential model: log(sigma_1) = a*t + b (implies sigma_1 ~ exp(a*t))
  - Fit polynomial model: log(sigma_1) = a*log(t) + b (implies sigma_1 ~ t^a)
  - Compare R^2 of each fit to determine which growth law dominates
  - Compute Pearson correlation between sigma_1(G) and sigma_1(W) trajectories
  - Compute Gini coefficient of the final SV spectrum

## Expected Outcomes

If the anisotropy-cascade / RG gauge-fixing model is correct:
1. SGD should show exponential sigma_1 growth (positive feedback loop active)
2. Muon should show sub-linear sigma_1 growth (feedback loop broken by orthogonalization)
3. The gradient-weight spectral correlation should be high for SGD and low for Muon
4. SGD should produce more anisotropic (higher Gini) spectra than Muon

## Critical Context from Prior Experiments
- **Exp 1.2b-i**: Muon is MORE chaotic (higher Lyapunov exponent) -- but chaos in *direction*, not magnitude
- **Exp 1.3a-i**: Per-layer effective rank stays higher for Muon (93.3% vs 89.5%) -- spectral democracy
- **Exp 1.3a-ii**: Muon's momentum buffer has 2x the effective rank of SGD's -- diverse update directions

## Step 1: Environment Setup

We use only NumPy (no PyTorch) to keep the experiment lightweight and transparent. All matrix
operations -- SVD, matrix multiplication, gradient computation -- are done explicitly so we can
inspect every intermediate quantity. The random seed is fixed for exact reproducibility.

In [ ]:
import numpy as np
import os
print(f"NumPy version: {np.__version__}")
print("Environment ready. Using pure NumPy for all linear algebra operations.")

In [ ]:
np.random.seed(42)
print("Random seed fixed to 42 for reproducibility.")

## Step 2: Experiment Configuration

**Design choices and their rationale:**

- **DIM = 32**: Large enough to have a meaningful singular value spectrum (32 singular values per
  layer) while remaining computationally cheap. The spectrum has enough degrees of freedom to
  distinguish isotropic from anisotropic distributions.
- **NUM_LAYERS = 4**: Deep enough that the product-of-matrices structure creates non-trivial
  gradient coupling between layers (gradients at layer i depend on all layers j != i), but
  shallow enough to avoid numerical instability.
- **NUM_STEPS = 500**: Long enough to observe growth trends and fit power laws, short enough
  to keep the experiment tractable.
- **BATCH_SIZE = 64**: Fixed batch (no stochasticity) -- this isolates the optimizer dynamics
  from gradient noise effects.
- **LR_MUON = 0.005**: Conservative learning rate for Muon. SGD's learning rate will be found
  automatically (max stable LR).
- **MOMENTUM = 0.9**: Standard momentum coefficient for both optimizers, ensuring the comparison
  is about the gradient processing (orthogonalization vs raw) rather than momentum strength.
- **NS_ITERS = 5**: Newton-Schulz iterations for Muon's orthogonalization. 5 iterations gives
  very good convergence to the true polar factor for 32x32 matrices.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
BATCH_SIZE = 64
LR_MUON = 0.005
MOMENTUM = 0.9
NS_ITERS = 5

print("Experiment configuration:")
print(f"  Matrix dimension:        {DIM}x{DIM} ({DIM**2} parameters per layer)")
print(f"  Number of layers:        {NUM_LAYERS} (total parameters: {NUM_LAYERS * DIM**2})")
print(f"  Training steps:          {NUM_STEPS}")
print(f"  Batch size:              {BATCH_SIZE} (fixed batch, no stochastic noise)")
print(f"  Muon learning rate:      {LR_MUON}")
print(f"  Momentum coefficient:    {MOMENTUM}")
print(f"  Newton-Schulz iterations:{NS_ITERS}")
print(f"  Singular values per layer: {DIM} (enough to measure Gini, erank, condition number)")

In [ ]:
# Random target matrix (fixed)
W_target = np.random.randn(DIM, DIM) * 0.5

# Inspect the target to understand the learning task
sv_target = np.linalg.svd(W_target, compute_uv=False)
print(f"Target matrix W_target: shape {W_target.shape}, Frobenius norm = {np.linalg.norm(W_target, 'fro'):.4f}")
print(f"  Target SV spectrum: max={sv_target[0]:.4f}, min={sv_target[-1]:.4f}, "
      f"condition number={sv_target[0]/sv_target[-1]:.2f}")
print(f"  Target is moderately anisotropic -- the network must learn to match this spectrum.")

In [ ]:
# Random input data (fixed batch)
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3

print(f"Input data X_data: shape {X_data.shape} ({DIM} features x {BATCH_SIZE} samples)")
print(f"  Input scale: mean={np.mean(np.abs(X_data)):.4f}, std={np.std(X_data):.4f}")
print(f"  Using a FIXED batch eliminates stochastic gradient noise --")
print(f"  any spectral bias we observe is purely from the optimizer dynamics, not sampling.")

In [ ]:
# Output directory
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Output directory: {SCRIPT_DIR}")

## Step 3: Core Functions -- Network, Loss, and Gradient Computation

### Deep Linear Network Architecture

The network computes: Y = W_L @ W_{L-1} @ ... @ W_1 @ X

This is a **deep linear network** -- no nonlinearities. Despite being linear in the end-to-end
mapping (the product W_L...W_1 is just another matrix), the optimization landscape is **non-convex**
due to the factored parameterization. This non-convexity is precisely what makes spectral dynamics
interesting:

- Gradients at each layer depend on all other layers (through the chain rule)
- The dominant singular direction of the gradient at layer i is coupled to the singular
  structure of layers j != i
- This coupling is what creates the feedback loop we are measuring

### Weight Initialization: Near-Identity

We initialize each weight as W_i = I + 0.1 * N(0,1). This means:
- Initial singular values cluster near 1.0 (slightly perturbed identity)
- Initial condition number is close to 1 (nearly isotropic)
- The product W_L...W_1 starts near identity
- Any growth in sigma_1 or condition number during training reflects optimizer dynamics,
  not initialization bias

In [ ]:
def init_weights(num_layers, seed=42):
    """Initialize layers near identity for stability."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

# Verify initialization properties
_test_weights = init_weights(NUM_LAYERS)
print("Initial weight properties (verifying near-identity initialization):")
for i, W in enumerate(_test_weights):
    sv = np.linalg.svd(W, compute_uv=False)
    print(f"  Layer {i}: sigma_1={sv[0]:.4f}, sigma_n={sv[-1]:.4f}, "
          f"kappa={sv[0]/sv[-1]:.4f}, Frobenius={np.linalg.norm(W, 'fro'):.4f}")
print(f"  All layers start with condition number near 1 -- nearly isotropic.")
print(f"  Any anisotropy growth during training is optimizer-induced.")
del _test_weights

In [ ]:
def forward(weights, X):
    """Forward pass: W_L @ ... @ W_1 @ X."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, target):
    """Loss = 0.5 * ||W_product @ X - T @ X||^2 / N."""
    pred = forward(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients(weights, X, target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    N = X.shape[1]

    # Forward pass storing activations
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())

    # Backward pass
    target_out = target @ X
    delta = (activations[-1] - target_out) / N

    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta

    return grads

### Newton-Schulz Orthogonalization: The Heart of Muon

The key difference between SGD and Muon is what happens to the gradient before it enters the
momentum buffer. In SGD, the raw gradient G is used directly. In Muon, G is replaced by its
**orthogonal polar factor** -- the closest orthogonal matrix to G.

Given the SVD of G = U @ diag(sigma) @ V^T, the polar factor is U @ V^T. This matrix has all
singular values equal to 1. The Newton-Schulz iteration is a fast, matrix-multiply-only method
to approximate this polar factor without explicitly computing the SVD:

    X_{k+1} = 1.5 * X_k - 0.5 * X_k @ (X_k^T @ X_k)

Starting from X_0 = G / ||G||_F, this converges cubically to U @ V^T.

**Why this breaks the feedback loop**: If the raw gradient G has sigma_1(G) >> sigma_2(G) (i.e.,
it points mostly along one direction), the orthogonalized version has sigma_1 = sigma_2 = ... = 1.
The update no longer preferentially amplifies the dominant direction. Every spectral direction
gets equal step size.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    """
    Newton-Schulz iteration to approximate the orthogonal polar factor.
    Returns closest orthogonal matrix to G (i.e., U @ V^T from SVD).
    """
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A

    return X

### Gini Coefficient: Measuring Spectral Concentration

The Gini coefficient (borrowed from economics) quantifies how "concentrated" a distribution is.
Applied to singular values:
- **Gini = 0**: All singular values are equal (perfectly isotropic / democratic spectrum)
- **Gini -> 1**: One singular value dominates (maximally anisotropic)

This complements the condition number (sigma_1/sigma_n) by being sensitive to the *full shape*
of the SV distribution, not just its extremes. A matrix could have sigma_1 = sigma_n but still
have an uneven interior -- the Gini catches this.

In [ ]:
def gini_coefficient(values):
    """
    Compute the Gini coefficient of a 1D array.
    0 = perfect equality, 1 = perfect inequality.
    """
    values = np.sort(np.abs(values))
    n = len(values)
    if n == 0 or np.sum(values) < 1e-30:
        return 0.0
    index = np.arange(1, n + 1)
    return (2.0 * np.sum(index * values) / (n * np.sum(values))) - (n + 1.0) / n

## Step 4: Optimizer Step Functions

### SGD with Momentum

The SGD update rule is:
- v_i <- momentum * v_i + G_i  (accumulate gradient into velocity)
- W_i <- W_i - lr * v_i  (step in the velocity direction)

The velocity v_i inherits the spectral structure of G_i. If G_i consistently has a dominant
singular direction aligned with sigma_1(W_i), then v_i amplifies this direction over time
through momentum accumulation. This is the **feedback loop** we are testing.

### Muon with Momentum

The Muon update rule replaces G_i with its orthogonal polar factor:
- G_ortho_i = NewtonSchulz(G_i)  (all singular values become 1)
- v_i <- momentum * v_i + G_ortho_i  (accumulate orthogonalized gradient)
- W_i <- W_i - lr * v_i  (step in the velocity direction)

Because G_ortho_i has uniform singular values, the velocity v_i does not preferentially
accumulate in any spectral direction. The momentum buffer itself may develop some anisotropy
(from the changing *directions* of successive orthogonal updates), but the magnitude of each
update is spectrally flat.

### Automatic SGD Learning Rate Selection

We search for the largest stable SGD learning rate by testing candidates from 0.05 down to
0.001, running 200 steps each, and checking for divergence. This ensures a fair comparison:
SGD gets its best stable learning rate, not an artificially small one.

In [ ]:
def find_stable_lr_sgd():
    """Find maximum stable SGD learning rate."""
    candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
    for lr in candidates:
        np.random.seed(42)
        weights = init_weights(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss(weights, X_data, W_target)
        stable = True
        for step in range(200):
            grads = compute_gradients(weights, X_data, W_target)
            for i in range(NUM_LAYERS):
                velocities[i] = MOMENTUM * velocities[i] + grads[i]
                weights[i] -= lr * velocities[i]
            loss = compute_loss(weights, X_data, W_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return 0.001

In [ ]:
def sgd_step(weights, velocities, lr):
    """One step of SGD with momentum. Returns (weights, velocities, raw_grads)."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        velocities[i] = MOMENTUM * velocities[i] + grads[i]
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities, grads

In [ ]:
def muon_step(weights, velocities, lr):
    """One step of Muon with momentum. Returns (weights, velocities, raw_grads)."""
    grads = compute_gradients(weights, X_data, W_target)
    for i in range(len(weights)):
        ortho_grad = newton_schulz_orthogonalize(grads[i])
        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
        weights[i] = weights[i] - lr * velocities[i]
    return weights, velocities, grads

## Step 5: Measurement Engine

The `run_and_measure` function is the core data collection loop. At **every single step** it
records:

1. **sigma_1(W_i)**: The top singular value of each weight matrix. This is the primary quantity
   of interest -- its growth rate (exponential vs sub-linear) is the experiment's central test.

2. **sigma_n(W_i)**: The bottom singular value. Together with sigma_1, this gives the condition
   number kappa = sigma_1/sigma_n, measuring spectral spread.

3. **sigma_1(G_i)**: The top singular value of the raw gradient at each layer. The correlation
   between this and sigma_1(W_i) measures whether the gradient "knows about" and reinforces
   the weight's dominant direction.

4. **Loss**: Standard training loss to verify both optimizers are actually learning.

5. **Final SV spectrum**: The complete set of 32 singular values at the last step, for
   computing Gini coefficients and bar-chart visualization.

The function also includes divergence detection -- if the loss exceeds 10^10 or becomes NaN,
it stops early and fills remaining entries with NaN.

In [ ]:
def run_and_measure(optimizer_name, optimizer_fn, lr, num_steps):
    """
    Run optimizer for num_steps and measure at EVERY step:
      - sigma_1(W_i) for each layer (top singular value)
      - sigma_n(W_i) for each layer (bottom singular value)
      - full SV spectrum at selected steps
      - sigma_1(G_i) for each layer (gradient top SV)
      - correlation between sigma_1(G_i) and sigma_1(W_i)
      - loss
    """
    np.random.seed(42)
    weights = init_weights(NUM_LAYERS)
    velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]

    # Storage: per step, per layer
    sigma1_W = np.zeros((num_steps + 1, NUM_LAYERS))  # top SV of weight
    sigman_W = np.zeros((num_steps + 1, NUM_LAYERS))   # bottom SV of weight
    sigma1_G = np.zeros((num_steps + 1, NUM_LAYERS))   # top SV of gradient
    losses = np.zeros(num_steps + 1)
    # Full SV spectrum at final step
    final_sv_spectrum = []

    # Measure at step 0
    for i in range(NUM_LAYERS):
        sv = np.linalg.svd(weights[i], compute_uv=False)
        sigma1_W[0, i] = sv[0]
        sigman_W[0, i] = sv[-1]
    losses[0] = compute_loss(weights, X_data, W_target)

    # Compute initial gradients for sigma1_G at step 0
    grads_init = compute_gradients(weights, X_data, W_target)
    for i in range(NUM_LAYERS):
        sv_g = np.linalg.svd(grads_init[i], compute_uv=False)
        sigma1_G[0, i] = sv_g[0]

    for step in range(1, num_steps + 1):
        weights, velocities, grads = optimizer_fn(weights, velocities, lr)

        # Measure
        for i in range(NUM_LAYERS):
            sv = np.linalg.svd(weights[i], compute_uv=False)
            sigma1_W[step, i] = sv[0]
            sigman_W[step, i] = sv[-1]

            sv_g = np.linalg.svd(grads[i], compute_uv=False)
            sigma1_G[step, i] = sv_g[0]

        loss = compute_loss(weights, X_data, W_target)
        losses[step] = loss

        # Check for divergence
        if np.isnan(loss) or loss > 1e10:
            print(f"    WARNING: {optimizer_name} diverged at step {step}!")
            # Fill remaining with NaN
            sigma1_W[step + 1:] = np.nan
            sigman_W[step + 1:] = np.nan
            sigma1_G[step + 1:] = np.nan
            losses[step + 1:] = np.nan
            break

    # Record final SV spectrum
    for i in range(NUM_LAYERS):
        sv = np.linalg.svd(weights[i], compute_uv=False)
        final_sv_spectrum.append(sv)

    return {
        'sigma1_W': sigma1_W,
        'sigman_W': sigman_W,
        'sigma1_G': sigma1_G,
        'losses': losses,
        'final_sv_spectrum': final_sv_spectrum,
    }

## Step 6: Growth Model Fitting

We fit two competing models to the sigma_1(W) trajectory for each layer:

### Model 1: Exponential Growth
    log(sigma_1) = a * t + b
    => sigma_1(t) ~ exp(a * t)

This is the **positive feedback prediction**: if the gradient consistently amplifies the
dominant direction, sigma_1 grows exponentially. The slope `a` is the exponential growth rate.
High R^2 means the growth is well-described by an exponential.

### Model 2: Polynomial (Sub-linear) Growth
    log(sigma_1) = a * log(t) + b
    => sigma_1(t) ~ t^a

This is the **bounded-step prediction**: if each update adds at most a fixed amount to sigma_1
(because the step is spectrally flat), sigma_1 grows as a power law. For Muon, we expect this
model to fit better, with exponent a < 1 (sub-linear growth).

### Decision Rule
We compare R^2 of each fit. Higher R^2 = better fit. If SGD prefers exponential and Muon
prefers polynomial, the feedback loop hypothesis is confirmed.

In [ ]:
def fit_exponential(steps, log_sigma1):
    """
    Fit log(sigma_1) = a*t + b  (exponential growth model).
    Returns (a, b, R^2).
    """
    # Filter out NaN/Inf
    mask = np.isfinite(log_sigma1)
    t = steps[mask].astype(float)
    y = log_sigma1[mask]
    if len(t) < 3:
        return 0.0, 0.0, 0.0

    # Linear regression: y = a*t + b
    A = np.vstack([t, np.ones(len(t))]).T
    result = np.linalg.lstsq(A, y, rcond=None)
    coeffs = result[0]
    a, b = coeffs[0], coeffs[1]

    y_pred = a * t + b
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-30 else 0.0

    return a, b, r2

In [ ]:
def fit_polynomial(steps, log_sigma1):
    """
    Fit log(sigma_1) = a*log(t) + b  (polynomial/power-law growth model).
    Returns (a, b, R^2).
    Only uses steps > 0 (since log(0) is undefined).
    """
    mask = np.isfinite(log_sigma1) & (steps > 0)
    t = steps[mask].astype(float)
    y = log_sigma1[mask]
    if len(t) < 3:
        return 0.0, 0.0, 0.0

    log_t = np.log(t)

    # Linear regression: y = a*log(t) + b
    A = np.vstack([log_t, np.ones(len(log_t))]).T
    result = np.linalg.lstsq(A, y, rcond=None)
    coeffs = result[0]
    a, b = coeffs[0], coeffs[1]

    y_pred = a * log_t + b
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 1e-30 else 0.0

    return a, b, r2

## Step 7: Run the Experiment

Now we execute both optimizers on the same initial conditions and collect all measurements.

**Sequence of operations:**
1. Find the maximum stable learning rate for SGD (automatic search)
2. Run SGD with momentum for 500 steps, recording sigma_1, sigma_n, sigma_1(G), and loss at every step
3. Run Muon for 500 steps with the same measurements
4. Both start from **identical** initial weights (same seed) to ensure the comparison is fair

Note: Both optimizers see the same fixed batch at every step. There is no stochasticity in
this experiment -- all observed differences are purely from the optimizer update rule.

In [ ]:
print("=" * 100)
print("1.3b-i-A: FEEDBACK MEASURE -- sigma_1(W) GROWTH RATE UNDER EACH OPTIMIZER")
print("=" * 100)
print(f"Setup: {NUM_LAYERS}-layer deep linear net (dim={DIM}), quadratic loss, {NUM_STEPS} steps")
print(f"Track: log(sigma_1(W)) vs step for each layer, for SGD and Muon")
print(f"Fit: exponential model log(s1) = a*t + b  vs  polynomial model log(s1) = a*log(t) + b")
print(f"Also: condition number sigma_1/sigma_n, Gini coefficient, corr(sigma_1(G), sigma_1(W))")
print(f"LR_Muon={LR_MUON}, Momentum={MOMENTUM}")
print("=" * 100)

In [ ]:
# Find stable SGD learning rate
lr_sgd = find_stable_lr_sgd()
print(f"\nSGD learning rate (max stable): {lr_sgd}")
print(f"Muon learning rate (fixed):     {LR_MUON}")
print(f"LR ratio (SGD/Muon):            {lr_sgd/LR_MUON:.1f}x")
print(f"\nNote: SGD typically needs a smaller LR because its raw gradients can have large")
print(f"singular values that amplify through momentum. Muon's orthogonalization bounds the")
print(f"update norm, allowing a more predictable step size.")

In [ ]:
# Initial loss
np.random.seed(42)
w_test = init_weights(NUM_LAYERS)
loss_init = compute_loss(w_test, X_data, W_target)
print(f"Initial loss: {loss_init:.6e}")
print(f"This is the squared error between (W_4 @ W_3 @ W_2 @ W_1 @ X) and (W_target @ X).")
print(f"Since weights start near identity, the initial prediction is roughly X itself,")
print(f"and the loss reflects how far the identity product is from W_target.")

In [ ]:
# Run both optimizers
print(f"\n{'=' * 100}")
print("RUNNING OPTIMIZERS AND TRACKING sigma_1 EVOLUTION")
print("=" * 100)

In [ ]:
print("\n  Running SGD...", flush=True)
results_sgd = run_and_measure('SGD', sgd_step, lr_sgd, NUM_STEPS)
print(f"    SGD final loss: {results_sgd['losses'][-1]:.6e}")
print(f"    SGD loss reduction: {loss_init / results_sgd['losses'][-1]:.1f}x")
sgd_final_s1 = [results_sgd['sigma1_W'][-1, i] for i in range(NUM_LAYERS)]
sgd_final_sn = [results_sgd['sigman_W'][-1, i] for i in range(NUM_LAYERS)]
sgd_final_kappa = [results_sgd['sigma1_W'][-1, i] / max(results_sgd['sigman_W'][-1, i], 1e-15) for i in range(NUM_LAYERS)]
print(f"    SGD final sigma_1 per layer: {['%.4f' % v for v in sgd_final_s1]}")
print(f"    SGD final sigma_n per layer: {['%.4f' % v for v in sgd_final_sn]}")
print(f"    SGD final condition number per layer: {['%.2f' % v for v in sgd_final_kappa]}")

In [ ]:
print("\n  Running Muon...", flush=True)
results_muon = run_and_measure('Muon', muon_step, LR_MUON, NUM_STEPS)
print(f"    Muon final loss: {results_muon['losses'][-1]:.6e}")
print(f"    Muon loss reduction: {loss_init / results_muon['losses'][-1]:.1f}x")
muon_final_s1 = [results_muon['sigma1_W'][-1, i] for i in range(NUM_LAYERS)]
muon_final_sn = [results_muon['sigman_W'][-1, i] for i in range(NUM_LAYERS)]
muon_final_kappa = [results_muon['sigma1_W'][-1, i] / max(results_muon['sigman_W'][-1, i], 1e-15) for i in range(NUM_LAYERS)]
print(f"    Muon final sigma_1 per layer: {['%.4f' % v for v in muon_final_s1]}")
print(f"    Muon final sigma_n per layer: {['%.4f' % v for v in muon_final_sn]}")
print(f"    Muon final condition number per layer: {['%.2f' % v for v in muon_final_kappa]}")

In [ ]:
steps = np.arange(NUM_STEPS + 1)

### Quick Sanity Check: Raw Run Summary

Before diving into the detailed analysis tables, examine the raw numbers above:

- **Loss reduction**: Both optimizers should reduce the loss substantially. If one barely moves,
  the spectral comparison is confounded by one optimizer simply not training. Check that both
  achieve meaningful loss reduction.
- **Final sigma_1 values**: If SGD's sigma_1 values are consistently larger than Muon's, that
  is a first hint that the feedback loop is active. But we need the growth *model* to confirm it.
- **Final condition numbers**: Large condition numbers for SGD but bounded for Muon would be
  consistent with SGD accumulating spectral anisotropy while Muon prevents it.

---

## Step 8: Detailed Analysis

We now analyze the collected data through six complementary lenses, each testing a different
aspect of the feedback loop hypothesis.

### Analysis 1: Growth Model Fitting -- Exponential vs Polynomial

This is the **central test** of the experiment. For each layer, we fit two models to the
log(sigma_1(W)) trajectory:

1. **Exponential**: log(sigma_1) = a*t + b. If R^2 is high, sigma_1 grows as exp(a*t).
   This indicates a positive feedback loop where the growth rate of sigma_1 is proportional
   to its current value.

2. **Polynomial**: log(sigma_1) = a*log(t) + b. If R^2 is high, sigma_1 grows as t^a.
   This indicates bounded per-step growth where sigma_1 increases by a fixed (or decreasing)
   amount each step.

**What we expect:**
- SGD: Exponential R^2 > Polynomial R^2 (feedback loop drives exponential growth)
- Muon: Polynomial R^2 > Exponential R^2 (spectrally flat updates produce sub-linear growth)

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 1: sigma_1(W) GROWTH MODEL FITS PER LAYER")
print("  Exponential model: log(sigma_1) = a*t + b   => sigma_1 ~ exp(a*t)")
print("  Polynomial model:  log(sigma_1) = a*log(t) + b  => sigma_1 ~ t^a")
print("  Better fit = higher R^2")
print("=" * 100)

In [ ]:
print(f"\n{'':>3} {'Layer':>5} | {'Exp a':>10} {'Exp R2':>8} | {'Poly a':>10} {'Poly R2':>8} | {'Best Fit':>10}")
print("-" * 75)

In [ ]:
sgd_exp_r2_all = []
sgd_poly_r2_all = []
muon_exp_r2_all = []
muon_poly_r2_all = []

In [ ]:
print("\n  SGD:")
for layer in range(NUM_LAYERS):
    log_s1 = np.log(results_sgd['sigma1_W'][:, layer] + 1e-30)
    a_exp, b_exp, r2_exp = fit_exponential(steps, log_s1)
    a_poly, b_poly, r2_poly = fit_polynomial(steps, log_s1)
    best = "EXPONENTIAL" if r2_exp > r2_poly else "POLYNOMIAL"
    sgd_exp_r2_all.append(r2_exp)
    sgd_poly_r2_all.append(r2_poly)
    print(f"  {'SGD':>3} {layer:5d} | {a_exp:10.6f} {r2_exp:8.4f} | {a_poly:10.6f} {r2_poly:8.4f} | {best:>10}")

In [ ]:
print("\n  Muon:")
for layer in range(NUM_LAYERS):
    log_s1 = np.log(results_muon['sigma1_W'][:, layer] + 1e-30)
    a_exp, b_exp, r2_exp = fit_exponential(steps, log_s1)
    a_poly, b_poly, r2_poly = fit_polynomial(steps, log_s1)
    best = "EXPONENTIAL" if r2_exp > r2_poly else "POLYNOMIAL"
    muon_exp_r2_all.append(r2_exp)
    muon_poly_r2_all.append(r2_poly)
    print(f"  {'Muon':>4} {layer:5d} | {a_exp:10.6f} {r2_exp:8.4f} | {a_poly:10.6f} {r2_poly:8.4f} | {best:>10}")

In [ ]:
sgd_mean_exp_r2 = np.mean(sgd_exp_r2_all)
sgd_mean_poly_r2 = np.mean(sgd_poly_r2_all)
muon_mean_exp_r2 = np.mean(muon_exp_r2_all)
muon_mean_poly_r2 = np.mean(muon_poly_r2_all)

In [ ]:
print(f"\n  SUMMARY:")
print(f"    SGD  mean R^2:  exponential={sgd_mean_exp_r2:.4f}   polynomial={sgd_mean_poly_r2:.4f}   "
      f"=> {'EXPONENTIAL' if sgd_mean_exp_r2 > sgd_mean_poly_r2 else 'POLYNOMIAL'} fits better")
print(f"    Muon mean R^2:  exponential={muon_mean_exp_r2:.4f}   polynomial={muon_mean_poly_r2:.4f}   "
      f"=> {'EXPONENTIAL' if muon_mean_exp_r2 > muon_mean_poly_r2 else 'POLYNOMIAL'} fits better")

#### Interpretation: Growth Model Fits

Examine the R^2 values above. Key questions to ask:

- **Does SGD prefer exponential?** If SGD's mean exponential R^2 > polynomial R^2, the feedback
  loop is producing self-reinforcing sigma_1 growth. The slope `a` in the exponential fit
  gives the growth rate: sigma_1 doubles every ln(2)/a steps.

- **Does Muon prefer polynomial?** If Muon's polynomial R^2 > exponential R^2, the
  orthogonalization is successfully preventing exponential amplification. The exponent `a`
  in the polynomial fit tells us the growth law: a < 1 means sub-linear, a = 1 means linear.

- **Layer-by-layer consistency?** If the pattern holds across all 4 layers, it is a robust
  property of the optimizer, not an artifact of a particular layer's gradient structure.

- **Caveat**: Both models might fit well if sigma_1 growth is modest -- over a short trajectory,
  exponential and polynomial can look similar. Pay attention to R^2 differences, not just
  which is larger.

### Analysis 2: Condition Number Evolution

The condition number kappa = sigma_1 / sigma_n measures the spectral spread of the weight
matrix. A well-conditioned matrix (kappa near 1) treats all directions equally. A poorly
conditioned matrix (large kappa) amplifies some directions and suppresses others.

**Why this matters for the feedback loop:**
- If the feedback loop is active (SGD), sigma_1 grows faster than sigma_n, so kappa increases
  monotonically -- the matrix becomes increasingly anisotropic.
- If the feedback loop is broken (Muon), all singular values receive similar updates, so
  kappa stays bounded -- spectral democracy is maintained.

We report kappa at steps 0, 50, 100, 200, 300, 500 to see the temporal evolution.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 2: CONDITION NUMBER sigma_1/sigma_n PER LAYER AT KEY STEPS")
print("=" * 100)

In [ ]:
report_steps = [0, 50, 100, 200, 300, 500]

In [ ]:
print(f"\n  {'Step':>6} | ", end="")
for layer in range(NUM_LAYERS):
    print(f"{'SGD L' + str(layer):>10} {'Muon L' + str(layer):>10} | ", end="")
print()
print("  " + "-" * (8 + (22 + 3) * NUM_LAYERS))

In [ ]:
for step in report_steps:
    if step > NUM_STEPS:
        continue
    print(f"  {step:6d} | ", end="")
    for layer in range(NUM_LAYERS):
        s1_sgd = results_sgd['sigma1_W'][step, layer]
        sn_sgd = results_sgd['sigman_W'][step, layer]
        kappa_sgd = s1_sgd / sn_sgd if sn_sgd > 1e-15 else np.inf

        s1_muon = results_muon['sigma1_W'][step, layer]
        sn_muon = results_muon['sigman_W'][step, layer]
        kappa_muon = s1_muon / sn_muon if sn_muon > 1e-15 else np.inf

        k_sgd_str = f"{kappa_sgd:10.2f}" if np.isfinite(kappa_sgd) else f"{'inf':>10}"
        k_muon_str = f"{kappa_muon:10.2f}" if np.isfinite(kappa_muon) else f"{'inf':>10}"
        print(f"{k_sgd_str} {k_muon_str} | ", end="")
    print()

#### Interpretation: Condition Number

Look at the temporal trajectory of kappa above:

- **Rate of growth**: Is SGD's kappa growing faster than Muon's? If so, the feedback loop is
  concentrating spectral energy into fewer directions for SGD.
- **Final ratio**: At step 500, how much larger is SGD's kappa than Muon's? A large gap
  confirms that SGD accumulates more anisotropy over the same number of steps.
- **Starting point**: Both should start near kappa = 1 (identity initialization). Any divergence
  from this starting point is optimizer-induced.

### Analysis 3: Gini Coefficient of the Final SV Spectrum

While the condition number only looks at the two extreme singular values, the Gini coefficient
captures the *entire distribution*. It measures inequality across all 32 singular values:

- **Gini = 0**: All 32 SVs are identical -- perfectly democratic spectrum
- **Gini near 1**: Almost all spectral energy is in one SV -- maximally concentrated

**Connection to the feedback loop:**
The feedback loop hypothesis predicts that SGD will have a higher Gini (more concentrated
spectrum) because the dominant direction receives disproportionate amplification. Muon should
have a lower Gini because its spectrally flat updates maintain democratic energy distribution.

This is closely related to the **effective rank** measured in Exp 1.3a-i, but Gini provides a
complementary perspective -- it is a single number that summarizes spectral inequality without
requiring a threshold or entropy computation.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 3: GINI COEFFICIENT OF SINGULAR VALUE SPECTRUM AT STEP 500")
print("  Gini = 0: all SVs equal (isotropic). Gini -> 1: one SV dominates (anisotropic).")
print("=" * 100)

In [ ]:
print(f"\n  {'Layer':>6} | {'SGD Gini':>10} | {'Muon Gini':>11} | {'SGD-Muon':>10}")
print("  " + "-" * 50)

In [ ]:
sgd_gini_all = []
muon_gini_all = []

In [ ]:
for layer in range(NUM_LAYERS):
    sgd_gini = gini_coefficient(results_sgd['final_sv_spectrum'][layer])
    muon_gini = gini_coefficient(results_muon['final_sv_spectrum'][layer])
    sgd_gini_all.append(sgd_gini)
    muon_gini_all.append(muon_gini)
    print(f"  {layer:6d} | {sgd_gini:10.4f} | {muon_gini:11.4f} | {sgd_gini - muon_gini:+10.4f}")

In [ ]:
sgd_mean_gini = np.mean(sgd_gini_all)
muon_mean_gini = np.mean(muon_gini_all)
print("  " + "-" * 50)
print(f"  {'MEAN':>6} | {sgd_mean_gini:10.4f} | {muon_mean_gini:11.4f} | {sgd_mean_gini - muon_mean_gini:+10.4f}")

#### Interpretation: Gini Coefficients

Check the SGD-Muon column above. A positive value means SGD is more anisotropic (higher Gini).
The magnitude tells us how much more concentrated SGD's spectrum is:
- A difference of 0.01-0.05 is modest but consistent
- A difference of 0.05-0.15 is substantial
- A difference > 0.15 is dramatic

Also check layer-by-layer consistency. If all 4 layers show SGD > Muon, the effect is robust.

### Analysis 4: Gradient-Weight Spectral Correlation -- The Feedback Loop Detector

This is the most **direct** test of the feedback loop mechanism. We compute the Pearson
correlation between sigma_1(G_i) (top SV of the gradient) and sigma_1(W_i) (top SV of the
weight) over the training trajectory.

**The feedback loop logic:**
1. If sigma_1(W) is large, the gradient G = (downstream terms) @ W^T @ ... concentrates
   energy along W's dominant direction, so sigma_1(G) is large too.
2. The large sigma_1(G) means the SGD update preferentially increases sigma_1(W).
3. Goto 1 -- positive feedback.

**Expected correlations:**
- **SGD: High positive correlation** (0.5 to 1.0). The gradient "knows about" the weight's
  anisotropy and reinforces it. sigma_1(G) and sigma_1(W) rise together.
- **Muon: Low or zero correlation**. The orthogonalization strips the gradient of magnitude
  information. Even if the raw gradient has large sigma_1(G), Muon's update has all SVs = 1,
  so it does not differentially amplify sigma_1(W). The correlation between the *raw* gradient's
  sigma_1 and the weight's sigma_1 may still be positive (the gradient structure is the same),
  but it should be weaker than SGD's because the orthogonalized updates do not reinforce the
  correlation.

We skip the first 10 steps to avoid initialization transients.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 4: CORRELATION corr(sigma_1(G), sigma_1(W)) PER LAYER")
print("  High correlation = gradient top direction tracks weight top direction = feedback loop ACTIVE")
print("  Low correlation = gradient spectrum decorrelated from weight spectrum = feedback loop BROKEN")
print("=" * 100)

In [ ]:
# Compute Pearson correlation over the training trajectory for each layer
# Use steps 10..500 to avoid initialization transients
start_step = 10

In [ ]:
print(f"\n  {'Layer':>6} | {'SGD corr':>10} | {'Muon corr':>11} | {'SGD-Muon':>10}")
print("  " + "-" * 50)

In [ ]:
sgd_corr_all = []
muon_corr_all = []

In [ ]:
for layer in range(NUM_LAYERS):
    # SGD
    s1_w_sgd = results_sgd['sigma1_W'][start_step:, layer]
    s1_g_sgd = results_sgd['sigma1_G'][start_step:, layer]
    mask_sgd = np.isfinite(s1_w_sgd) & np.isfinite(s1_g_sgd)
    if np.sum(mask_sgd) > 2:
        corr_sgd = np.corrcoef(s1_w_sgd[mask_sgd], s1_g_sgd[mask_sgd])[0, 1]
    else:
        corr_sgd = np.nan
    sgd_corr_all.append(corr_sgd)

    # Muon
    s1_w_muon = results_muon['sigma1_W'][start_step:, layer]
    s1_g_muon = results_muon['sigma1_G'][start_step:, layer]
    mask_muon = np.isfinite(s1_w_muon) & np.isfinite(s1_g_muon)
    if np.sum(mask_muon) > 2:
        corr_muon = np.corrcoef(s1_w_muon[mask_muon], s1_g_muon[mask_muon])[0, 1]
    else:
        corr_muon = np.nan
    muon_corr_all.append(corr_muon)

    print(f"  {layer:6d} | {corr_sgd:10.4f} | {corr_muon:11.4f} | {corr_sgd - corr_muon:+10.4f}")

In [ ]:
sgd_mean_corr = np.nanmean(sgd_corr_all)
muon_mean_corr = np.nanmean(muon_corr_all)
print("  " + "-" * 50)
print(f"  {'MEAN':>6} | {sgd_mean_corr:10.4f} | {muon_mean_corr:11.4f} | {sgd_mean_corr - muon_mean_corr:+10.4f}")

#### Interpretation: Gradient-Weight Correlation

The SGD-Muon difference column is crucial:

- **Positive difference (SGD > Muon)**: Confirms the feedback loop hypothesis. SGD's gradient
  spectral structure is more tightly coupled to the weight spectral structure.
- **Magnitude of difference**: A difference of 0.2+ is strong evidence. Even 0.1 is meaningful
  given the deterministic setup.
- **Negative correlations**: If Muon shows negative correlation, it means sigma_1(G) tends to
  *decrease* when sigma_1(W) increases -- the orthogonalization may be actively counter-acting
  the feedback loop, not just passively breaking it.
- **Both positive but SGD higher**: This is the most likely outcome. Even for Muon, the raw
  gradient (which we measure sigma_1(G) from) still reflects weight structure. The difference
  is that Muon does not *act on* this anisotropy.

### Analysis 5: sigma_1(W) Trajectory at Key Steps

A direct view of the sigma_1 values at selected timesteps. This table lets you visually
inspect the growth pattern:
- **Exponential growth** would show values increasing faster and faster (gaps between
  consecutive rows widen)
- **Linear growth** would show constant gaps
- **Sub-linear growth** would show gaps that shrink over time

Compare SGD vs Muon columns for each layer to see which pattern each optimizer follows.

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 5: sigma_1(W) VALUES AT KEY STEPS")
print("=" * 100)

In [ ]:
print(f"\n  {'Step':>6} | ", end="")
for layer in range(NUM_LAYERS):
    print(f"{'SGD L' + str(layer):>8} {'Muon L' + str(layer):>8} | ", end="")
print()
print("  " + "-" * (8 + (18 + 3) * NUM_LAYERS))

In [ ]:
for step in report_steps:
    if step > NUM_STEPS:
        continue
    print(f"  {step:6d} | ", end="")
    for layer in range(NUM_LAYERS):
        s1_sgd = results_sgd['sigma1_W'][step, layer]
        s1_muon = results_muon['sigma1_W'][step, layer]
        print(f"{s1_sgd:8.3f} {s1_muon:8.3f} | ", end="")
    print()

### Analysis 6: Exponential Growth Rate Comparison

The slope `a` from the exponential fit log(sigma_1) = a*t + b directly measures the
**exponential growth rate** of the top singular value. This is the most compact summary
of the feedback loop strength:

- **SGD slope >> Muon slope**: SGD's feedback loop drives faster sigma_1 growth
- **Ratio SGD/Muon > 1**: Quantifies how many times stronger the feedback loop is for SGD
- **Both slopes positive**: Both optimizers may increase sigma_1, but the question is whether
  SGD does it *exponentially* (constant positive slope) vs Muon doing it *sub-linearly*
  (slope that decreases if we were to fit it on a moving window)

In [ ]:
print(f"\n\n{'=' * 100}")
print("TABLE 6: GROWTH RATE d(log sigma_1)/dt  (exponential fit slope)")
print("  Positive = sigma_1 growing. Larger magnitude = faster growth.")
print("=" * 100)

In [ ]:
print(f"\n  {'Layer':>6} | {'SGD slope':>12} | {'Muon slope':>12} | {'Ratio SGD/Muon':>15}")
print("  " + "-" * 55)

In [ ]:
sgd_slopes = []
muon_slopes = []

In [ ]:
for layer in range(NUM_LAYERS):
    log_s1_sgd = np.log(results_sgd['sigma1_W'][:, layer] + 1e-30)
    a_sgd, _, _ = fit_exponential(steps, log_s1_sgd)
    sgd_slopes.append(a_sgd)

    log_s1_muon = np.log(results_muon['sigma1_W'][:, layer] + 1e-30)
    a_muon, _, _ = fit_exponential(steps, log_s1_muon)
    muon_slopes.append(a_muon)

    ratio = a_sgd / a_muon if abs(a_muon) > 1e-10 else np.inf
    ratio_str = f"{ratio:15.2f}" if np.isfinite(ratio) else f"{'N/A':>15}"
    print(f"  {layer:6d} | {a_sgd:12.6f} | {a_muon:12.6f} | {ratio_str}")

In [ ]:
sgd_mean_slope = np.mean(sgd_slopes)
muon_mean_slope = np.mean(muon_slopes)
ratio_mean = sgd_mean_slope / muon_mean_slope if abs(muon_mean_slope) > 1e-10 else np.inf
ratio_str = f"{ratio_mean:15.2f}" if np.isfinite(ratio_mean) else f"{'N/A':>15}"
print("  " + "-" * 55)
print(f"  {'MEAN':>6} | {sgd_mean_slope:12.6f} | {muon_mean_slope:12.6f} | {ratio_str}")

#### Interpretation: Growth Rates

The mean ratio in the table above quantifies the feedback loop asymmetry. If SGD's slope is
N times larger than Muon's, the feedback loop amplifies sigma_1 growth by a factor of N.

Note that the absolute magnitudes of the slopes are also informative:
- Very small slopes (< 10^-4) mean sigma_1 barely changes -- neither optimizer drives
  significant anisotropy growth
- Moderate slopes (10^-3 to 10^-2) mean sigma_1 roughly doubles every few hundred steps
- Large slopes (> 10^-2) mean rapid anisotropy accumulation

---

## Step 9: Visualization

Six-panel diagnostic plot covering all key aspects of the experiment:

- **(a) log(sigma_1) vs Step**: The primary growth curve. Linear = exponential growth.
  Concave-down = sub-linear growth. Compare SGD (blue) vs Muon (red).
- **(b) sigma_1 vs Step (linear scale)**: Same data without log transform. Exponential growth
  shows up as an upward-curving line; sub-linear as a flattening curve.
- **(c) Condition Number vs Step**: How the spectral spread evolves. Log-scale y-axis to
  accommodate potentially large SGD condition numbers.
- **(d) Rolling Correlation**: A 50-step rolling window of corr(sigma_1(G), sigma_1(W)).
  Shows how the feedback loop strength varies during training.
- **(e) Loss vs Step**: Verification that both optimizers are training. Log-scale.
- **(f) Final SV Spectrum (Layer 0)**: Bar chart comparing the full 32-SV spectrum at the
  final step. SGD should have a more peaked distribution (one large bar); Muon should be
  more uniform.

In [ ]:
print(f"\n\n{'=' * 100}")
print("GENERATING PLOTS")
print("=" * 100)

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('1.3b-i-A: Feedback Measure -- sigma_1(W) Growth Rate\n'
                 f'{NUM_LAYERS}-layer linear net, dim={DIM}, {NUM_STEPS} steps',
                 fontsize=14, fontweight='bold')

    colors_sgd = ['#1f77b4', '#4a9fd4', '#7ec7f0', '#a8d8f8']
    colors_muon = ['#d62728', '#e74c3c', '#f08080', '#f4a6a6']

    # --- Panel (a): log(sigma_1) vs step ---
    ax = axes[0, 0]
    ax.set_title('(a) log(sigma_1(W)) vs Step')
    for layer in range(NUM_LAYERS):
        log_s1_sgd = np.log(results_sgd['sigma1_W'][:, layer] + 1e-30)
        log_s1_muon = np.log(results_muon['sigma1_W'][:, layer] + 1e-30)
        ax.plot(steps, log_s1_sgd, color=colors_sgd[layer % len(colors_sgd)],
                linewidth=1.5, label=f'SGD L{layer}' if layer == 0 else None, alpha=0.7)
        ax.plot(steps, log_s1_muon, color=colors_muon[layer % len(colors_muon)],
                linewidth=1.5, label=f'Muon L{layer}' if layer == 0 else None,
                linestyle='--', alpha=0.7)
    # Plot mean with bold lines
    sgd_mean_log_s1 = np.mean(np.log(results_sgd['sigma1_W'] + 1e-30), axis=1)
    muon_mean_log_s1 = np.mean(np.log(results_muon['sigma1_W'] + 1e-30), axis=1)
    ax.plot(steps, sgd_mean_log_s1, 'b-', linewidth=3, label='SGD (mean)')
    ax.plot(steps, muon_mean_log_s1, 'r--', linewidth=3, label='Muon (mean)')
    ax.set_xlabel('Step')
    ax.set_ylabel('log(sigma_1)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # --- Panel (b): sigma_1(W) vs step (linear scale) ---
    ax = axes[0, 1]
    ax.set_title('(b) sigma_1(W) vs Step (linear scale)')
    for layer in range(NUM_LAYERS):
        ax.plot(steps, results_sgd['sigma1_W'][:, layer],
                color=colors_sgd[layer % len(colors_sgd)], linewidth=1.2, alpha=0.7)
        ax.plot(steps, results_muon['sigma1_W'][:, layer],
                color=colors_muon[layer % len(colors_muon)], linewidth=1.2, alpha=0.7,
                linestyle='--')
    sgd_mean_s1 = np.mean(results_sgd['sigma1_W'], axis=1)
    muon_mean_s1 = np.mean(results_muon['sigma1_W'], axis=1)
    ax.plot(steps, sgd_mean_s1, 'b-', linewidth=3, label='SGD (mean)')
    ax.plot(steps, muon_mean_s1, 'r--', linewidth=3, label='Muon (mean)')
    ax.set_xlabel('Step')
    ax.set_ylabel('sigma_1(W)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (c): Condition number over time ---
    ax = axes[0, 2]
    ax.set_title('(c) Condition Number sigma_1/sigma_n vs Step')
    for layer in range(NUM_LAYERS):
        kappa_sgd = results_sgd['sigma1_W'][:, layer] / np.maximum(results_sgd['sigman_W'][:, layer], 1e-15)
        kappa_muon = results_muon['sigma1_W'][:, layer] / np.maximum(results_muon['sigman_W'][:, layer], 1e-15)
        ax.semilogy(steps, kappa_sgd, color=colors_sgd[layer % len(colors_sgd)],
                     linewidth=1.2, alpha=0.7)
        ax.semilogy(steps, kappa_muon, color=colors_muon[layer % len(colors_muon)],
                     linewidth=1.2, alpha=0.7, linestyle='--')
    # Mean condition number
    kappa_sgd_mean = np.mean(
        results_sgd['sigma1_W'] / np.maximum(results_sgd['sigman_W'], 1e-15), axis=1)
    kappa_muon_mean = np.mean(
        results_muon['sigma1_W'] / np.maximum(results_muon['sigman_W'], 1e-15), axis=1)
    ax.semilogy(steps, kappa_sgd_mean, 'b-', linewidth=3, label='SGD (mean)')
    ax.semilogy(steps, kappa_muon_mean, 'r--', linewidth=3, label='Muon (mean)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Condition Number')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (d): corr(sigma_1(G), sigma_1(W)) rolling window ---
    ax = axes[1, 0]
    ax.set_title('(d) Rolling corr(sigma_1(G), sigma_1(W))\nwindow=50 steps')
    window = 50
    for layer in range(NUM_LAYERS):
        rolling_corr_sgd = []
        rolling_corr_muon = []
        for t in range(window, NUM_STEPS + 1):
            s1w = results_sgd['sigma1_W'][t - window:t, layer]
            s1g = results_sgd['sigma1_G'][t - window:t, layer]
            if np.std(s1w) > 1e-12 and np.std(s1g) > 1e-12:
                rolling_corr_sgd.append(np.corrcoef(s1w, s1g)[0, 1])
            else:
                rolling_corr_sgd.append(0)

            s1w_m = results_muon['sigma1_W'][t - window:t, layer]
            s1g_m = results_muon['sigma1_G'][t - window:t, layer]
            if np.std(s1w_m) > 1e-12 and np.std(s1g_m) > 1e-12:
                rolling_corr_muon.append(np.corrcoef(s1w_m, s1g_m)[0, 1])
            else:
                rolling_corr_muon.append(0)

        t_axis = np.arange(window, NUM_STEPS + 1)
        ax.plot(t_axis, rolling_corr_sgd,
                color=colors_sgd[layer % len(colors_sgd)], linewidth=1, alpha=0.5)
        ax.plot(t_axis, rolling_corr_muon,
                color=colors_muon[layer % len(colors_muon)], linewidth=1, alpha=0.5,
                linestyle='--')

    # Compute mean rolling corr
    all_rc_sgd = np.zeros(NUM_STEPS + 1 - window)
    all_rc_muon = np.zeros(NUM_STEPS + 1 - window)
    for layer in range(NUM_LAYERS):
        for idx, t in enumerate(range(window, NUM_STEPS + 1)):
            s1w = results_sgd['sigma1_W'][t - window:t, layer]
            s1g = results_sgd['sigma1_G'][t - window:t, layer]
            if np.std(s1w) > 1e-12 and np.std(s1g) > 1e-12:
                all_rc_sgd[idx] += np.corrcoef(s1w, s1g)[0, 1]
            s1w_m = results_muon['sigma1_W'][t - window:t, layer]
            s1g_m = results_muon['sigma1_G'][t - window:t, layer]
            if np.std(s1w_m) > 1e-12 and np.std(s1g_m) > 1e-12:
                all_rc_muon[idx] += np.corrcoef(s1w_m, s1g_m)[0, 1]
    all_rc_sgd /= NUM_LAYERS
    all_rc_muon /= NUM_LAYERS
    t_axis = np.arange(window, NUM_STEPS + 1)
    ax.plot(t_axis, all_rc_sgd, 'b-', linewidth=3, label='SGD (mean)')
    ax.plot(t_axis, all_rc_muon, 'r--', linewidth=3, label='Muon (mean)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Pearson Correlation')
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # --- Panel (e): Gini coefficient over time ---
    ax = axes[1, 1]
    ax.set_title('(e) Gini Coefficient of SV Spectrum vs Step')
    # Compute Gini at every 10 steps
    gini_steps = np.arange(0, NUM_STEPS + 1, 10)
    sgd_gini_ts = np.zeros((len(gini_steps), NUM_LAYERS))
    muon_gini_ts = np.zeros((len(gini_steps), NUM_LAYERS))

    # We need to re-run to get full SV spectra at intermediate steps
    # Instead, approximate Gini from sigma1/sigman ratio
    # Actually, we only stored sigma1 and sigman, not full spectrum at each step.
    # We can compute Gini from just sigma1 and sigman as an approximation,
    # or we use the tracked data more cleverly.
    # For a cleaner approach: plot sigma1/mean_sigma as a measure of anisotropy
    # which is related to Gini. We have sigma1 and sigman tracked.
    # Let's just plot sigma1/sigman (condition number) which captures the spread.
    # But we already did that in panel (c). Instead let's plot the loss curves.
    ax_loss = ax
    ax_loss.set_title('(e) Loss vs Step')
    ax_loss.semilogy(steps, results_sgd['losses'], 'b-', linewidth=2.5, label='SGD')
    ax_loss.semilogy(steps, results_muon['losses'], 'r--', linewidth=2.5, label='Muon')
    ax_loss.set_xlabel('Step')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=9)
    ax_loss.grid(True, alpha=0.3)

    # --- Panel (f): Final SV spectrum bar chart ---
    ax = axes[1, 2]
    ax.set_title(f'(f) Final SV Spectrum (Layer 0, step {NUM_STEPS})')
    sv_sgd_l0 = results_sgd['final_sv_spectrum'][0]
    sv_muon_l0 = results_muon['final_sv_spectrum'][0]
    x_idx = np.arange(DIM)
    width = 0.35
    ax.bar(x_idx - width / 2, sv_sgd_l0, width, color='blue', alpha=0.6, label='SGD')
    ax.bar(x_idx + width / 2, sv_muon_l0, width, color='red', alpha=0.6, label='Muon')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('Singular Value')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plot_path = os.path.join(SCRIPT_DIR, 'sigma1_growth_rate.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved to: {plot_path}")

In [ ]:
except ImportError:
    print("\n  WARNING: matplotlib not available, skipping plots.")
    plot_path = None

## Step 10: Hypothesis Testing and Final Verdict

We now evaluate 5 pre-registered tests and a composite test that together assess the
feedback loop hypothesis:

| Test | Description | Pass Condition |
|------|-------------|---------------|
| T1 | SGD growth is more exponential than Muon's | SGD exp-R^2 > Muon exp-R^2 |
| T2 | Muon growth is better described by polynomial than exponential | Muon poly-R^2 > Muon exp-R^2 |
| T3 | SGD growth IS exponential (not polynomial) | SGD exp-R^2 > SGD poly-R^2 |
| T4 | Feedback loop is active for SGD, broken for Muon | SGD corr > Muon corr |
| T5 | SGD has more anisotropic spectrum | SGD Gini > Muon Gini |
| **Composite** | SGD exponential + Muon sub-linear | T1 AND T2 |

**Decision thresholds:**
- 4-5 tests pass + composite confirmed = **PASS** (strong evidence for feedback loop)
- 3 tests pass = **PARTIAL PASS** (suggestive but not conclusive)
- 2 tests pass = **WEAK SIGNAL** (possible but confounded)
- 0-1 tests pass = **FAIL** (feedback loop hypothesis not supported)

In [ ]:
print(f"\n\n{'=' * 100}")
print("FINAL VERDICT: sigma_1(W) GROWTH RATE FEEDBACK LOOP")
print("=" * 100)

In [ ]:
# Test 1: SGD exponential fit R^2 > Muon exponential fit R^2
# (SGD growth is better described by exponential)
test1_pass = sgd_mean_exp_r2 > muon_mean_exp_r2

In [ ]:
# Test 2: Muon polynomial fit R^2 > Muon exponential fit R^2
# (Muon growth is better described by sub-linear/polynomial)
test2_pass = muon_mean_poly_r2 > muon_mean_exp_r2

In [ ]:
# Test 3: SGD exponential fit R^2 > SGD polynomial fit R^2
# (SGD growth IS exponential, not polynomial)
test3_pass = sgd_mean_exp_r2 > sgd_mean_poly_r2

In [ ]:
# Test 4: SGD corr(sigma1_G, sigma1_W) > Muon corr(sigma1_G, sigma1_W)
# (feedback loop is active for SGD, broken for Muon)
test4_pass = sgd_mean_corr > muon_mean_corr

In [ ]:
# Test 5: SGD Gini > Muon Gini at final step
# (SGD has more anisotropic spectrum)
test5_pass = sgd_mean_gini > muon_mean_gini

In [ ]:
# Composite: SGD exponential > Muon sub-linear
# This is the key claim: R2_exp(SGD) > R2_exp(Muon) AND R2_poly(Muon) > R2_exp(Muon)
composite_pass = test1_pass and test2_pass

In [ ]:
tests = [test1_pass, test2_pass, test3_pass, test4_pass, test5_pass]
tests_passed = sum(tests)
tests_total = 5

In [ ]:
print(f"""
  MEASURED QUANTITIES:
  ---------------------------------------------------------------
  Growth Model Fit (mean R^2 across layers):
    SGD:   exponential R^2 = {sgd_mean_exp_r2:.4f}   polynomial R^2 = {sgd_mean_poly_r2:.4f}
    Muon:  exponential R^2 = {muon_mean_exp_r2:.4f}   polynomial R^2 = {muon_mean_poly_r2:.4f}

  Feedback Loop Correlation corr(sigma_1(G), sigma_1(W)):
    SGD:   {sgd_mean_corr:.4f}
    Muon:  {muon_mean_corr:.4f}

  Gini Coefficient of SV Spectrum at step {NUM_STEPS}:
    SGD:   {sgd_mean_gini:.4f}
    Muon:  {muon_mean_gini:.4f}

  sigma_1 exponential growth rate (slope of log(s1) vs t):
    SGD:   {sgd_mean_slope:.6f} per step
    Muon:  {muon_mean_slope:.6f} per step
  ---------------------------------------------------------------

  HYPOTHESIS CHECKS:
  ---------------------------------------------------------------
  T1: SGD exp-R^2 > Muon exp-R^2 (SGD growth is more exponential)
      SGD: {sgd_mean_exp_r2:.4f} vs Muon: {muon_mean_exp_r2:.4f}
      -> {"CONFIRMED" if test1_pass else "REJECTED"}

  T2: Muon poly-R^2 > Muon exp-R^2 (Muon growth is sub-linear)
      poly: {muon_mean_poly_r2:.4f} vs exp: {muon_mean_exp_r2:.4f}
      -> {"CONFIRMED" if test2_pass else "REJECTED"}

  T3: SGD exp-R^2 > SGD poly-R^2 (SGD growth IS exponential)
      exp: {sgd_mean_exp_r2:.4f} vs poly: {sgd_mean_poly_r2:.4f}
      -> {"CONFIRMED" if test3_pass else "REJECTED"}

  T4: SGD corr > Muon corr (feedback loop active for SGD, broken for Muon)
      SGD: {sgd_mean_corr:.4f} vs Muon: {muon_mean_corr:.4f}
      -> {"CONFIRMED" if test4_pass else "REJECTED"}

  T5: SGD Gini > Muon Gini (SGD has more anisotropic spectrum)
      SGD: {sgd_mean_gini:.4f} vs Muon: {muon_mean_gini:.4f}
      -> {"CONFIRMED" if test5_pass else "REJECTED"}

  COMPOSITE: SGD exponential > Muon sub-linear
      (T1 AND T2): {"CONFIRMED" if composite_pass else "REJECTED"}
  ---------------------------------------------------------------
""")

In [ ]:
if tests_passed >= 4 and composite_pass:
    overall = "PASS"
    detail = (
        f"  {tests_passed}/5 tests pass + composite confirmed.\n"
        "  SGD sigma_1 growth is exponential (positive feedback loop).\n"
        "  Muon sigma_1 growth is sub-linear (bounded step size breaks feedback).\n"
        "  Gradient-weight spectral correlation confirms: SGD feedback loop active,\n"
        "  Muon feedback loop broken by orthogonalization."
    )
elif tests_passed >= 3:
    overall = "PARTIAL PASS"
    detail = (
        f"  {tests_passed}/5 tests pass, composite={'PASS' if composite_pass else 'FAIL'}.\n"
        f"  T1 (SGD exp > Muon exp):    {'PASS' if test1_pass else 'FAIL'}\n"
        f"  T2 (Muon poly > Muon exp):  {'PASS' if test2_pass else 'FAIL'}\n"
        f"  T3 (SGD exp > SGD poly):    {'PASS' if test3_pass else 'FAIL'}\n"
        f"  T4 (SGD corr > Muon corr):  {'PASS' if test4_pass else 'FAIL'}\n"
        f"  T5 (SGD Gini > Muon Gini):  {'PASS' if test5_pass else 'FAIL'}"
    )
elif tests_passed >= 2:
    overall = "WEAK SIGNAL"
    detail = (
        f"  {tests_passed}/5 tests pass, composite={'PASS' if composite_pass else 'FAIL'}.\n"
        f"  T1 (SGD exp > Muon exp):    {'PASS' if test1_pass else 'FAIL'}\n"
        f"  T2 (Muon poly > Muon exp):  {'PASS' if test2_pass else 'FAIL'}\n"
        f"  T3 (SGD exp > SGD poly):    {'PASS' if test3_pass else 'FAIL'}\n"
        f"  T4 (SGD corr > Muon corr):  {'PASS' if test4_pass else 'FAIL'}\n"
        f"  T5 (SGD Gini > Muon Gini):  {'PASS' if test5_pass else 'FAIL'}"
    )
else:
    overall = "FAIL"
    detail = (
        f"  Only {tests_passed}/5 tests pass, composite={'PASS' if composite_pass else 'FAIL'}.\n"
        f"  T1 (SGD exp > Muon exp):    {'PASS' if test1_pass else 'FAIL'}\n"
        f"  T2 (Muon poly > Muon exp):  {'PASS' if test2_pass else 'FAIL'}\n"
        f"  T3 (SGD exp > SGD poly):    {'PASS' if test3_pass else 'FAIL'}\n"
        f"  T4 (SGD corr > Muon corr):  {'PASS' if test4_pass else 'FAIL'}\n"
        f"  T5 (SGD Gini > Muon Gini):  {'PASS' if test5_pass else 'FAIL'}"
    )

In [ ]:
print(f"""
  +========================================================================+
  |  VERDICT: {overall:<63}|
  +========================================================================+
  |                                                                        |""")
for line in detail.split('\n'):
    print(f"  |  {line:<70}|")
print(f"""  |                                                                        |
  +========================================================================+
""")

In [ ]:
print("=" * 100)
print(f"  Tests passed: {tests_passed}/{tests_total}")
print(f"  Composite (SGD exponential > Muon sub-linear): {'PASS' if composite_pass else 'FAIL'}")
print(f"  Overall: {overall}")
print("=" * 100)

## Conclusions and Implications

### What This Experiment Measures

This experiment directly tests the **anisotropy self-amplification hypothesis**: that SGD with
momentum creates a positive feedback loop where the gradient's spectral structure reinforces the
weight's existing anisotropy, leading to exponential growth of the dominant singular value.
Meanwhile, Muon's Newton-Schulz orthogonalization breaks this loop by projecting every gradient
onto the orthogonal manifold, ensuring spectrally flat updates.

### Connections to the RG Gauge-Fixing Framework

In the renormalization group (RG) perspective on Muon:
- SGD's feedback loop is analogous to a **relevant operator** in the RG flow: small perturbations
  in the dominant singular direction grow exponentially under iteration, driving the system away
  from the fixed point (identity matrix / isotropic spectrum).
- Muon's orthogonalization acts as a **gauge-fixing constraint**: by projecting onto the orthogonal
  group at each step, it removes the redundant degrees of freedom (singular value magnitudes of
  the gradient) that drive the feedback loop. This is analogous to fixing a gauge in field theory
  to remove unphysical modes.
- The bounded sigma_1 growth under Muon corresponds to **marginal or irrelevant perturbations** in
  the RG sense: the orthogonal projection keeps the dynamics on a lower-dimensional manifold where
  anisotropy cannot self-amplify.

### Implications for Practice

If the feedback loop is confirmed:
1. **SGD in deep networks** may waste capacity by concentrating spectral energy in few directions
2. **Muon's advantage** is not just faster convergence but qualitatively different dynamics:
   maintaining spectral democracy throughout training
3. **Learning rate sensitivity**: SGD's exponential sigma_1 growth explains why it is more
   sensitive to learning rate -- a slightly too-large LR accelerates the feedback loop,
   causing divergence. Muon's bounded growth makes it inherently more stable.

### Follow-up Experiments
- **1.3b-i-B**: Does the feedback loop direction align with the Hessian's top eigenvector?
- **1.3b-ii-A**: Does the Hessian spectrum reflect the weight anisotropy created by SGD?
- **1.3c-i**: Can we isolate the spectral growth rate from the learning rate confound?